Functions

Functions are the backbone of every Python program. They let you **name a reusable block of logic**, break complex problems into manageable pieces, and write code you can read, test, and maintain. This notebook goes from the basics all the way to advanced patterns like closures, decorators, and generators.

## What You'll Learn
1. Defining and calling functions
2. Parameters & arguments — positional, keyword, defaults
3. `*args` and `**kwargs` — variadic functions
4. Return values
5. Scope — local, enclosing, global, built-in (LEGB)
6. `lambda` — anonymous functions
7. Higher-order functions — `map`, `filter`, `sorted`, `functools`
8. Closures
9. Decorators
10. Generators & `yield`
11. Type hints & docstrings
12. Recursion

---

## 1. Defining & Calling Functions

A function is defined with `def`, given a name, and called by appending `()`.

In [1]:
# Simplest possible function
def greet():
    print("Hello, World!")

greet()   # call the function

Hello, World!


In [2]:
# Functions are objects — you can assign them, pass them, store them
def say_hello():
    print("Hello!")

# Assign to another variable
hi = say_hello
hi()   # same function, different name

# Inspect the function
print(type(say_hello))          # <class 'function'>
print(say_hello.__name__)       # say_hello
print(callable(say_hello))      # True
print(callable(42))             # False

Hello!
<class 'function'>
say_hello
True
False


In [3]:
# Functions without an explicit return statement return None
def no_return():
    x = 1 + 1   # does work but returns nothing

result = no_return()
print(result)           # None
print(type(result))     # <class 'NoneType'>

None
<class 'NoneType'>


---

## 2. Parameters & Arguments

**Parameters** are the names in the function definition. **Arguments** are the values you pass when calling.

In [4]:
# Positional parameters — order matters
def greet(name, greeting):
    print(f"{greeting}, {name}!")

greet("Alice", "Hello")     # positional — order matters
greet("Bob",   "Howdy")

Hello, Alice!
Howdy, Bob!


In [5]:
# Keyword arguments — name them explicitly, order doesn't matter
greet(greeting="Hi", name="Carol")   # reversed, still works
greet("Dave", greeting="Hey")        # mix positional + keyword

Hi, Carol!
Hey, Dave!


In [6]:
# Default parameter values
def greet(name, greeting="Hello"):
    print(f"{greeting}, {name}!")

greet("Alice")             # uses default greeting
greet("Bob", "Good day")   # overrides default

# ⚠️ Parameters with defaults must come AFTER those without
# def bad(greeting="Hello", name):  ← SyntaxError!
# def good(name, greeting="Hello"): ← OK

Hello, Alice!
Good day, Bob!


In [7]:
# ⚠️ The mutable default argument trap — a classic Python gotcha!
# Default values are evaluated ONCE when the function is defined,
# not each time it's called. Using a mutable default (list, dict)
# causes them to be SHARED across all calls.

def bad_append(item, container=[]):   # ← dangerous!
    container.append(item)
    return container

print(bad_append(1))   # [1]   — looks fine
print(bad_append(2))   # [1, 2] — wait, where did 1 come from?!
print(bad_append(3))   # [1, 2, 3] — the default list persists!

print()

# ✅ Fix: use None as the default, create inside the function
def good_append(item, container=None):
    if container is None:
        container = []       # fresh list every call
    container.append(item)
    return container

print(good_append(1))   # [1]
print(good_append(2))   # [2] — correct!

[1]
[1, 2]
[1, 2, 3]

[1]
[2]


In [8]:
# Positional-only and keyword-only parameters (Python 3.8+)
# /  separates positional-only from the rest
# *  separates the rest from keyword-only

def strict(pos_only, /, normal, *, kw_only):
    print(f"pos_only={pos_only}, normal={normal}, kw_only={kw_only}")

strict(1, 2, kw_only=3)           # OK
strict(1, normal=2, kw_only=3)    # OK

try:
    strict(pos_only=1, normal=2, kw_only=3)  # ← pos_only can't be keyword
except TypeError as e:
    print(f"Error: {e}")

try:
    strict(1, 2, 3)               # ← kw_only must be keyword
except TypeError as e:
    print(f"Error: {e}")

pos_only=1, normal=2, kw_only=3
pos_only=1, normal=2, kw_only=3
Error: strict() got some positional-only arguments passed as keyword arguments: 'pos_only'
Error: strict() takes 2 positional arguments but 3 were given


---

## 3. `*args` and `**kwargs` — Variadic Functions

These let functions accept any number of arguments.

In [9]:
# *args — collect extra positional arguments into a TUPLE
def total(*args):
    print(f"args = {args}, type = {type(args)}")
    return sum(args)

print(total(1, 2, 3))          # 6
print(total(10, 20, 30, 40))   # 100
print(total())                 # 0 — empty tuple

args = (1, 2, 3), type = <class 'tuple'>
6
args = (10, 20, 30, 40), type = <class 'tuple'>
100
args = (), type = <class 'tuple'>
0


In [10]:
# Mixing regular params with *args
def greet_many(greeting, *names):
    for name in names:
        print(f"{greeting}, {name}!")

greet_many("Hello", "Alice", "Bob", "Carol")

print()

# Unpacking a list/tuple into positional arguments with *
numbers = [1, 2, 3, 4, 5]
print(total(*numbers))      # same as total(1, 2, 3, 4, 5)

Hello, Alice!
Hello, Bob!
Hello, Carol!

args = (1, 2, 3, 4, 5), type = <class 'tuple'>
15


In [11]:
# **kwargs — collect extra keyword arguments into a DICT
def print_info(**kwargs):
    print(f"kwargs = {kwargs}, type = {type(kwargs)}")
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

print_info(name="Alice", age=30, city="New York")

kwargs = {'name': 'Alice', 'age': 30, 'city': 'New York'}, type = <class 'dict'>
  name: Alice
  age: 30
  city: New York


In [12]:
# Unpacking a dict into keyword arguments with **
def create_user(name, age, role="user"):
    return {"name": name, "age": age, "role": role}

data = {"name": "Bob", "age": 25, "role": "admin"}
user = create_user(**data)   # same as create_user(name="Bob", age=25, role="admin")
print(user)

{'name': 'Bob', 'age': 25, 'role': 'admin'}


In [13]:
# Full signature: positional, *args, keyword-only, **kwargs
def everything(a, b, *args, flag=False, **kwargs):
    print(f"a={a}, b={b}")
    print(f"args={args}")
    print(f"flag={flag}")
    print(f"kwargs={kwargs}")

everything(1, 2, 3, 4, 5, flag=True, color="blue", size=10)

a=1, b=2
args=(3, 4, 5)
flag=True
kwargs={'color': 'blue', 'size': 10}


In [14]:
# Practical use: pass-through / wrapper functions
def log_call(func, *args, **kwargs):
    """Call func with any args/kwargs and log the result."""
    print(f"Calling {func.__name__}({args}, {kwargs})")
    result = func(*args, **kwargs)
    print(f"  → returned: {result}")
    return result

log_call(max, 3, 1, 4, 1, 5, 9, 2, 6)
log_call(sorted, [3, 1, 2], reverse=True)

Calling max((3, 1, 4, 1, 5, 9, 2, 6), {})
  → returned: 9
Calling sorted(([3, 1, 2],), {'reverse': True})
  → returned: [3, 2, 1]


[3, 2, 1]

---

## 4. Return Values

In [15]:
# Single return value
def square(n):
    return n ** 2

print(square(5))   # 25

# Early return — exit as soon as we have the answer
def is_even(n):
    if n % 2 == 0:
        return True
    return False

# Even cleaner:
def is_even(n):
    return n % 2 == 0

print(is_even(4), is_even(7))

25
True False


In [16]:
# Multiple return values — Python returns a tuple
def min_max(numbers):
    return min(numbers), max(numbers)   # actually returns a tuple

result = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(result)          # (1, 9)
print(type(result))    # <class 'tuple'>

# Unpack directly
lo, hi = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(f"Min: {lo}, Max: {hi}")

(1, 9)
<class 'tuple'>
Min: 1, Max: 9


In [17]:
# Returning a dict for named results — more readable than a tuple
def stats(numbers):
    n = len(numbers)
    return {
        "count": n,
        "sum":   sum(numbers),
        "mean":  sum(numbers) / n,
        "min":   min(numbers),
        "max":   max(numbers),
    }

data = [10, 20, 30, 40, 50]
s = stats(data)
for key, value in s.items():
    print(f"  {key:6}: {value}")

  count : 5
  sum   : 150
  mean  : 30.0
  min   : 10
  max   : 50


In [18]:
# Guard clauses — return early to avoid deep nesting

# ❌ Heavily nested (hard to read)
def process_order_bad(order):
    if order is not None:
        if order.get("items"):
            if order.get("paid"):
                return f"Shipping {len(order['items'])} items"
            else:
                return "Order not paid"
        else:
            return "No items in order"
    else:
        return "No order"

# ✅ Guard clauses (flat and readable)
def process_order(order):
    if order is None:
        return "No order"
    if not order.get("items"):
        return "No items in order"
    if not order.get("paid"):
        return "Order not paid"
    return f"Shipping {len(order['items'])} items"

orders = [
    None,
    {"items": [], "paid": True},
    {"items": ["book", "pen"], "paid": False},
    {"items": ["laptop"], "paid": True},
]
for o in orders:
    print(process_order(o))

No order
No items in order
Order not paid
Shipping 1 items


---

## 5. Scope — LEGB Rule

When Python looks up a name, it searches four scopes in order:
**L**ocal → **E**nclosing → **G**lobal → **B**uilt-in

In [19]:
# Local scope — variables created inside a function
x = "global"

def demo():
    x = "local"     # creates a NEW local variable — doesn't touch global x
    print(f"Inside:  x = {x}")

demo()
print(f"Outside: x = {x}")   # global x unchanged

Inside:  x = local
Outside: x = global


In [20]:
# Reading a global from inside a function is fine
PI = 3.14159

def circle_area(r):
    return PI * r ** 2    # reads global PI

print(circle_area(5))

# Modifying a global requires the 'global' keyword
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(f"counter = {counter}")   # 2

# ⚠️ Avoid global when possible — it makes code harder to reason about.
# Prefer returning values or using a class.

78.53975
counter = 2


In [21]:
# Enclosing scope — nested functions
def outer():
    message = "from outer"

    def inner():
        print(message)    # reads from enclosing scope

    inner()

outer()

# Modifying an enclosing variable requires 'nonlocal'
def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter = make_counter()
print(counter())   # 1
print(counter())   # 2
print(counter())   # 3

from outer
1
2
3


In [22]:
# Built-in scope — Python's built-in names
# len, print, range, type, sum, min, max, etc. live here

print(len("hello"))   # 'len' found in built-in scope

# ⚠️ Don't shadow built-ins with your own variables!
# list = [1, 2, 3]   ← now 'list' no longer refers to the built-in type!
# print(list([4, 5]))  ← TypeError!

import builtins
print(dir(builtins))   # see all built-in names

5
['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarni

---

## 6. `lambda` — Anonymous Functions

`lambda` creates a small, throwaway function in a single expression. Best used for short callbacks — not complex logic.

In [23]:
# lambda syntax: lambda parameters: expression
square = lambda x: x ** 2
add    = lambda x, y: x + y
greet  = lambda name: f"Hello, {name}!"

print(square(5))
print(add(3, 4))
print(greet("Alice"))

# Equivalent def versions
def square(x):   return x ** 2
def add(x, y):   return x + y

25
7
Hello, Alice!


In [24]:
# Most common use: as a key function for sorting
people = [
    {"name": "Alice", "age": 30},
    {"name": "Bob",   "age": 25},
    {"name": "Carol", "age": 35},
    {"name": "Dave",  "age": 28},
]

# Sort by age
by_age = sorted(people, key=lambda p: p["age"])
for p in by_age:
    print(f"  {p['name']:8} {p['age']}")

print()

# Sort strings by length, then alphabetically
words = ["banana", "fig", "apple", "kiwi", "cherry"]
print(sorted(words, key=lambda w: (len(w), w)))

  Bob      25
  Dave     28
  Alice    30
  Carol    35

['fig', 'kiwi', 'apple', 'banana', 'cherry']


In [25]:
# Lambda with conditionals
classify = lambda n: "even" if n % 2 == 0 else "odd"
print([classify(n) for n in range(6)])

# ⚠️ If your lambda spans multiple expressions or needs logic,
# use a regular def — it's more readable and debuggable.

['even', 'odd', 'even', 'odd', 'even', 'odd']


---

## 7. Higher-Order Functions

A **higher-order function** either takes a function as an argument, returns a function, or both. Python has several built-in ones.

In [26]:
# map(func, iterable) — apply a function to every element
numbers = [1, 2, 3, 4, 5]

squares  = list(map(lambda n: n ** 2, numbers))
as_strs  = list(map(str, numbers))

print(squares)
print(as_strs)

# map over two lists simultaneously
a = [1, 2, 3]
b = [10, 20, 30]
sums = list(map(lambda x, y: x + y, a, b))
print(sums)

# Note: list comprehensions are usually preferred over map()
squares = [n ** 2 for n in numbers]   # same result, more Pythonic

[1, 4, 9, 16, 25]
['1', '2', '3', '4', '5']
[11, 22, 33]


In [27]:
# filter(func, iterable) — keep only elements where func returns True
numbers = range(20)

evens  = list(filter(lambda n: n % 2 == 0, numbers))
odds   = list(filter(lambda n: n % 2 != 0, numbers))

print(f"Evens: {evens}")
print(f"Odds:  {odds}")

# With a real function
words = ["hello", "", "world", "", "python"]
non_empty = list(filter(None, words))   # None filters out falsy values
print(non_empty)

# Again — comprehension equivalent is usually preferred
evens = [n for n in range(20) if n % 2 == 0]

Evens: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
Odds:  [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
['hello', 'world', 'python']


In [28]:
# functools.reduce() — collapse a sequence to a single value
from functools import reduce

numbers = [1, 2, 3, 4, 5]

total   = reduce(lambda acc, n: acc + n, numbers)       # sum
product = reduce(lambda acc, n: acc * n, numbers)       # product
maximum = reduce(lambda a, b: a if a > b else b, numbers)  # max

print(f"Sum:     {total}")
print(f"Product: {product}")
print(f"Max:     {maximum}")

Sum:     15
Product: 120
Max:     5


In [29]:
# functools.partial() — pre-fill some arguments of a function
from functools import partial

def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube   = partial(power, exponent=3)
double = partial(pow, 2)    # 2^n — using built-in pow

print(square(5))    # 25
print(cube(3))      # 27
print([double(n) for n in range(8)])  # powers of 2

25
27
[1, 2, 4, 8, 16, 32, 64, 128]


In [30]:
# functools.lru_cache — memoize expensive function results
from functools import lru_cache
import time

@lru_cache(maxsize=None)   # cache unlimited results
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

start = time.perf_counter()
print(fibonacci(50))
elapsed = time.perf_counter() - start
print(f"Computed in {elapsed*1000:.3f}ms")
print(f"Cache info: {fibonacci.cache_info()}")

12586269025
Computed in 0.107ms
Cache info: CacheInfo(hits=48, misses=51, maxsize=None, currsize=51)


---

## 8. Closures

A **closure** is a function that **remembers the environment in which it was created** — it carries variables from its enclosing scope with it, even after that scope is gone.

In [31]:
# A factory function that returns a closure
def make_multiplier(factor):
    def multiply(n):
        return n * factor   # 'factor' is captured from make_multiplier's scope
    return multiply         # return the inner function

double = make_multiplier(2)
triple = make_multiplier(3)
times10 = make_multiplier(10)

print(double(5))    # 10
print(triple(5))    # 15
print(times10(5))   # 50

# Each closure has its own 'factor'
print(double.__closure__[0].cell_contents)   # 2
print(triple.__closure__[0].cell_contents)   # 3

10
15
50
2
3


In [32]:
# Closures with state — a counter without a class
def make_counter(start=0, step=1):
    count = start

    def counter():
        nonlocal count
        current = count
        count += step
        return current

    return counter

c1 = make_counter()         # 0, 1, 2, ...
c2 = make_counter(10, 5)    # 10, 15, 20, ...

print([c1() for _ in range(5)])
print([c2() for _ in range(5)])
print(f"c1 and c2 are independent: c1()={c1()}, c2()={c2()}")

[0, 1, 2, 3, 4]
[10, 15, 20, 25, 30]
c1 and c2 are independent: c1()=5, c2()=35


In [33]:
# ⚠️ Classic closure-in-a-loop gotcha
# All closures share the SAME variable from the enclosing scope

funcs_bad = [lambda: i for i in range(5)]   # captures variable 'i', not its value
print([f() for f in funcs_bad])   # [4, 4, 4, 4, 4] — all see the final i=4!

# ✅ Fix 1: capture the value with a default argument
funcs_good = [lambda i=i: i for i in range(5)]
print([f() for f in funcs_good])  # [0, 1, 2, 3, 4] — correct!

# ✅ Fix 2: use partial
from functools import partial
funcs_partial = [partial(lambda i: i, i) for i in range(5)]
print([f() for f in funcs_partial])  # [0, 1, 2, 3, 4]

[4, 4, 4, 4, 4]
[0, 1, 2, 3, 4]
[0, 1, 2, 3, 4]


---

## 9. Decorators

A **decorator** is a function that **wraps another function** to extend or modify its behavior — without changing the original function's source code. The `@` syntax is syntactic sugar.

In [34]:
# Building a decorator from scratch
def shout(func):
    """Decorator that uppercases the return value."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)   # call the original
        return result.upper()            # modify the result
    return wrapper

# Applying it manually:
def greet(name):
    return f"hello, {name}"

greet = shout(greet)   # ← this is what @shout does!
print(greet("Alice"))

# Using the @ syntax (equivalent and cleaner):
@shout
def farewell(name):
    return f"goodbye, {name}"

print(farewell("Bob"))

HELLO, ALICE
GOODBYE, BOB


In [35]:
# Using functools.wraps to preserve the original function's metadata
from functools import wraps

def shout(func):
    @wraps(func)              # ← always use this inside a decorator!
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    """Return a friendly greeting."""
    return f"hello, {name}"

print(greet("Alice"))
print(greet.__name__)   # 'greet' (not 'wrapper' — thanks to @wraps)
print(greet.__doc__)    # docstring preserved

HELLO, ALICE
greet
Return a friendly greeting.


In [36]:
# Practical decorator 1: timing
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed*1000:.2f}ms")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))
print(slow_sum(5_000_000))

slow_sum took 9.63ms
499999500000
slow_sum took 47.46ms
12499997500000


In [37]:
# Practical decorator 2: retry on failure
import random
from functools import wraps

def retry(times=3):
    """Decorator factory — takes arguments."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  Attempt {attempt}/{times} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

@retry(times=4)
def flaky_operation():
    """Fails 70% of the time."""
    if random.random() < 0.7:
        raise ConnectionError("Network blip")
    return "Success!"

random.seed(42)
try:
    print(flaky_operation())
except ConnectionError:
    print("All attempts failed.")

  Attempt 1/4 failed: Network blip
  Attempt 2/4 failed: Network blip
  Attempt 3/4 failed: Network blip
  Attempt 4/4 failed: Network blip
All attempts failed.


In [38]:
# Stacking decorators — applied bottom-up
from functools import wraps

def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return "**" + func(*args, **kwargs) + "**"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return "_" + func(*args, **kwargs) + "_"
    return wrapper

@bold       # applied second
@italic     # applied first
def message():
    return "hello"

print(message())   # **_hello_**
# Equivalent to: bold(italic(message))()

**_hello_**


---

## 10. Generators & `yield`

A **generator** is a function that uses `yield` to produce values **one at a time**, on demand. It pauses between yields and resumes where it left off — making it extremely memory-efficient.

In [39]:
# A basic generator function
def count_up(start, stop):
    n = start
    while n <= stop:
        yield n      # pause here, send n to the caller
        n += 1       # resume here next time

gen = count_up(1, 5)
print(type(gen))         # <class 'generator'>

print(next(gen))         # 1
print(next(gen))         # 2
print(next(gen))         # 3

# Iterate over the rest
for n in gen:
    print(n, end=" ")
print()

<class 'generator'>
1
2
3
4 5 


In [40]:
# Memory comparison: list vs generator
import sys

N = 1_000_000
lst = [n ** 2 for n in range(N)]     # all values in memory NOW
gen = (n ** 2 for n in range(N))     # values computed on demand

print(f"List size:      {sys.getsizeof(lst):>12,} bytes")
print(f"Generator size: {sys.getsizeof(gen):>12,} bytes")

# Generators are perfect when you only need to iterate once
total = sum(n ** 2 for n in range(N))   # never builds a list
print(f"Sum: {total:,}")

List size:         8,448,728 bytes
Generator size:          208 bytes
Sum: 333,332,833,333,500,000


In [41]:
# Infinite generators — only possible because values are lazy
def naturals(start=1):
    """Infinite sequence of natural numbers."""
    n = start
    while True:
        yield n
        n += 1

def take(n, iterable):
    """Take the first n values from an iterable."""
    for _, value in zip(range(n), iterable):
        yield value

print(list(take(10, naturals())))
print(list(take(5, naturals(100))))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[100, 101, 102, 103, 104]


In [42]:
# yield from — delegate to a sub-generator
def chain(*iterables):
    """Concatenate multiple iterables."""
    for it in iterables:
        yield from it    # delegate to each iterable

result = list(chain([1, 2, 3], "abc", range(5, 8)))
print(result)

# Flattening a nested structure with yield from
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # recurse into sublists
        else:
            yield item

data = [1, [2, 3], [4, [5, 6]], 7, [8, [9, [10]]]]
print(list(flatten(data)))

[1, 2, 3, 'a', 'b', 'c', 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [43]:
# Generators as pipelines — each stage processes one item at a time
import re

# Simulate a large log file
log_lines = [
    "2026-05-17 ERROR Failed to connect to database",
    "2026-05-17 INFO  Server started on port 8080",
    "2026-05-17 ERROR Timeout waiting for response",
    "2026-05-17 DEBUG Received 42 bytes",
    "2026-05-17 ERROR Disk space critically low",
    "2026-05-17 INFO  Request completed in 120ms",
]

# Pipeline stages — each is a generator
def only_errors(lines):
    for line in lines:
        if "ERROR" in line:
            yield line

def extract_message(lines):
    for line in lines:
        yield line.split(" ERROR ")[-1].strip()

def uppercase(lines):
    for line in lines:
        yield line.upper()

# Compose the pipeline — nothing runs until we consume it
pipeline = uppercase(extract_message(only_errors(log_lines)))

print("Error messages:")
for msg in pipeline:
    print(f"  {msg}")

Error messages:
  FAILED TO CONNECT TO DATABASE
  TIMEOUT WAITING FOR RESPONSE
  DISK SPACE CRITICALLY LOW


---

## 11. Type Hints & Docstrings

Type hints (PEP 484) make functions self-documenting and enable static analysis tools. Docstrings explain *what* a function does.

In [44]:
from typing import Optional, Union

# Type hints in function signatures
def greet(name: str, times: int = 1) -> str:
    return (f"Hello, {name}! " * times).strip()

print(greet("Alice"))
print(greet("Bob", 3))

# Python 3.10+: use | instead of Union
def process(value: int | float | str) -> str:
    return str(value)

# Optional[X] means X | None
def find_user(user_id: int) -> Optional[dict]:
    users = {1: {"name": "Alice"}, 2: {"name": "Bob"}}
    return users.get(user_id)

print(find_user(1))
print(find_user(99))

Hello, Alice!
Hello, Bob! Hello, Bob! Hello, Bob!
{'name': 'Alice'}
None


In [45]:
# Common type hint patterns
from typing import Any

def example(
    a: int,
    b: float,
    c: str,
    d: bool,
    e: list[int],           # list of ints
    f: dict[str, int],      # dict mapping str → int
    g: tuple[int, str],     # tuple with exactly (int, str)
    h: tuple[int, ...],     # tuple of any number of ints
    i: set[str],            # set of strings
    j: int | None,          # int or None
    k: Any,                 # any type
) -> None:
    pass

# Callable type hint
from typing import Callable

def apply(func: Callable[[int], str], value: int) -> str:
    return func(value)

print(apply(str, 42))

42


In [46]:
# Docstrings — three formats in common use

# 1. Google style (popular, readable)
def divide(a: float, b: float) -> float:
    """Divide a by b.

    Args:
        a: The numerator.
        b: The denominator. Must not be zero.

    Returns:
        The quotient of a and b.

    Raises:
        ZeroDivisionError: If b is zero.

    Examples:
        >>> divide(10, 2)
        5.0
        >>> divide(7, 3)
        2.3333333333333335
    """
    if b == 0:
        raise ZeroDivisionError("b must not be zero")
    return a / b

print(divide(10, 4))
help(divide)   # renders the docstring nicely

2.5
Help on function divide in module __main__:

divide(a: float, b: float) -> float
    Divide a by b.

    Args:
        a: The numerator.
        b: The denominator. Must not be zero.

    Returns:
        The quotient of a and b.

    Raises:
        ZeroDivisionError: If b is zero.

    Examples:
        >>> divide(10, 2)
        5.0
        >>> divide(7, 3)
        2.3333333333333335



In [47]:
# Introspecting functions at runtime
import inspect

def sample(x: int, y: float = 1.0, *args, flag: bool = False, **kwargs) -> str:
    """A sample function for introspection."""
    pass

sig = inspect.signature(sample)
print(f"Signature:  {sig}")
print(f"Parameters:")
for name, param in sig.parameters.items():
    print(f"  {name:8} kind={param.kind.name:20} default={param.default!r}")

print(f"\nAnnotations: {sample.__annotations__}")
print(f"Docstring:   {sample.__doc__}")

Signature:  (x: int, y: float = 1.0, *args, flag: bool = False, **kwargs) -> str
Parameters:
  x        kind=POSITIONAL_OR_KEYWORD default=<class 'inspect._empty'>
  y        kind=POSITIONAL_OR_KEYWORD default=1.0
  args     kind=VAR_POSITIONAL       default=<class 'inspect._empty'>
  flag     kind=KEYWORD_ONLY         default=False
  kwargs   kind=VAR_KEYWORD          default=<class 'inspect._empty'>

Annotations: {'x': <class 'int'>, 'y': <class 'float'>, 'flag': <class 'bool'>, 'return': <class 'str'>}
Docstring:   A sample function for introspection.


---

## 12. Recursion

A **recursive function** calls itself to solve a problem by breaking it into smaller instances of the same problem. Every recursive function needs a **base case** to stop.

In [48]:
# Classic: factorial
def factorial(n: int) -> int:
    """Return n! (n factorial)."""
    if n <= 1:          # base case
        return 1
    return n * factorial(n - 1)   # recursive case

for n in range(8):
    print(f"{n}! = {factorial(n)}")

0! = 1
1! = 1
2! = 2
3! = 6
4! = 24
5! = 120
6! = 720
7! = 5040


In [49]:
# Fibonacci — with and without memoization
def fib_slow(n):
    """Exponential time — recomputes everything."""
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

from functools import lru_cache

@lru_cache(maxsize=None)
def fib_fast(n):
    """Linear time — caches every subproblem."""
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

import time

start = time.perf_counter(); fib_slow(32); slow = time.perf_counter() - start
start = time.perf_counter(); fib_fast(32); fast = time.perf_counter() - start

print(f"fib_slow(32) = {fib_slow(32):,}  in {slow*1000:.1f}ms")
print(f"fib_fast(32) = {fib_fast(32):,}  in {fast*1000:.3f}ms")
print(f"Speedup: {slow/fast:.0f}x")

fib_slow(32) = 2,178,309  in 235.9ms
fib_fast(32) = 2,178,309  in 0.052ms
Speedup: 4530x


In [50]:
# Recursion on nested data structures
def deep_sum(data):
    """Sum all numbers in an arbitrarily nested list."""
    total = 0
    for item in data:
        if isinstance(item, list):
            total += deep_sum(item)   # recurse
        else:
            total += item
    return total

nested = [1, [2, 3], [4, [5, [6, 7]]], 8]
print(deep_sum(nested))   # 36

# Recursively count items in a nested dict
def count_leaves(obj):
    """Count non-dict values in a nested dict."""
    if not isinstance(obj, dict):
        return 1
    return sum(count_leaves(v) for v in obj.values())

config = {
    "server": {"host": "localhost", "port": 8080},
    "db": {"host": "db.local", "name": "mydb", "pool": {"size": 10}},
    "debug": True,
}
print(f"Config has {count_leaves(config)} leaf values")

36
Config has 6 leaf values


In [51]:
# Python's recursion limit — and how to work around it
import sys

print(f"Default recursion limit: {sys.getrecursionlimit()}")

# You can raise it, but deep recursion risks a stack overflow
# sys.setrecursionlimit(10000)

# ✅ For deep recursion, convert to iteration with an explicit stack
def deep_sum_iterative(data):
    """Iterative version — no recursion limit issues."""
    stack = list(data)
    total = 0
    while stack:
        item = stack.pop()
        if isinstance(item, list):
            stack.extend(item)
        else:
            total += item
    return total

print(deep_sum_iterative([1, [2, 3], [4, [5, [6, 7]]], 8]))  # 36

Default recursion limit: 3000
36


---

## 13. Quick Reference

| Concept | Syntax / Tool | Use When |
|---|---|---|
| Define a function | `def name(params):` | Anytime you have reusable logic |
| Default argument | `def f(x, y=10):` | Parameter has a sensible default |
| Variadic positional | `def f(*args):` | Unknown number of positional args |
| Variadic keyword | `def f(**kwargs):` | Unknown number of keyword args |
| Unpack into call | `f(*list)`, `f(**dict)` | Spread a collection as arguments |
| Anonymous function | `lambda x: x*2` | Short callbacks (sort key, map) |
| Preserve metadata | `@functools.wraps` | Inside every decorator |
| Pre-fill args | `functools.partial` | Specialise a general function |
| Memoize | `@functools.lru_cache` | Expensive, pure recursive functions |
| Lazy values | `yield` / generator | Large sequences, pipelines |
| Closure | inner `def` + `nonlocal` | Stateful functions without classes |
| Decorator | `@decorator` | Cross-cutting: logging, timing, retry |
| Type hints | `def f(x: int) -> str:` | Docs, IDE support, static analysis |

In [52]:
# 🏆 Practice: Build a mini function toolkit

from functools import wraps, lru_cache
import time

# 1. A decorator that validates all arguments are positive numbers
def require_positive(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        all_args = list(args) + list(kwargs.values())
        for arg in all_args:
            if isinstance(arg, (int, float)) and arg <= 0:
                raise ValueError(f"{func.__name__}: all numeric args must be positive, got {arg}")
        return func(*args, **kwargs)
    return wrapper

# 2. A memoized recursive function
@lru_cache(maxsize=128)
def triangle(n: int) -> int:
    """Return the nth triangle number: 1+2+3+...+n."""
    if n <= 1:
        return n
    return n + triangle(n - 1)

# 3. A generator that applies functions as a pipeline
def pipeline(*funcs):
    """Return a function that applies funcs left to right."""
    def apply(value):
        for func in funcs:
            value = func(value)
        return value
    return apply

# 4. A closure-based accumulator
def make_accumulator(initial=0):
    total = initial
    def add(n):
        nonlocal total
        total += n
        return total
    return add

# --- Demo ---
print("Triangle numbers:")
print([triangle(n) for n in range(1, 11)])

print("\nPipeline (strip → lower → split):")
process = pipeline(str.strip, str.lower, str.split)
print(process("  Hello World Python  "))

print("\nAccumulator:")
acc = make_accumulator()
for n in [10, 20, 30, 5]:
    print(f"  add({n}) → {acc(n)}")

print("\nRequire positive:")
@require_positive
def volume(l, w, h):
    return l * w * h

print(f"  volume(3, 4, 5) = {volume(3, 4, 5)}")
try:
    volume(3, -1, 5)
except ValueError as e:
    print(f"  Error: {e}")

Triangle numbers:
[1, 3, 6, 10, 15, 21, 28, 36, 45, 55]

Pipeline (strip → lower → split):
['hello', 'world', 'python']

Accumulator:
  add(10) → 10
  add(20) → 30
  add(30) → 60
  add(5) → 65

Require positive:
  volume(3, 4, 5) = 60
  Error: volume: all numeric args must be positive, got -1
